# Compare two harnesses on a coding task

Give each harness its own broken calculator, then check the fixes with the same tests. Both use the same OpenAI model.

Run the cells in order. You need an [OpenAI API key](https://platform.openai.com/api-keys) with API credit.

[Open in Colab](https://colab.research.google.com/github/BerriAI/liteagents/blob/main/cookbook/compare_harnesses/compare.ipynb)

## 1. Install

Install LiteAgents and the integrations used in this notebook.

In [ ]:
%pip install -q --progress-bar off "liteagents[pydantic-ai,claude-sdk] @ https://github.com/BerriAI/liteagents/releases/download/v0.3.0a6/liteagents-0.3.0a6-py3-none-any.whl"

## 2. Add your key

Run this cell, paste your key into the hidden input, and press Enter.

In [ ]:
import os
from getpass import getpass

os.environ["OPENAI_API_KEY"] = (os.environ.get("OPENAI_API_KEY") or getpass("OpenAI API key: ")).strip()
if not os.environ["OPENAI_API_KEY"]:
    raise ValueError("Run this cell again and enter your OpenAI API key.")

## 3. Make a small coding task

The subtraction function adds instead. These tests describe the correct behavior.

In [ ]:
import tempfile
from pathlib import Path

workspace = Path(tempfile.mkdtemp(prefix="liteagents-"))

CALCULATOR = """def subtract(left, right):
    return left + right
"""
TESTS = """import unittest
from calculator import subtract

class SubtractionTests(unittest.TestCase):
    def test_positive(self):
        self.assertEqual(subtract(9, 4), 5)
    def test_negative(self):
        self.assertEqual(subtract(-3, 7), -10)
    def test_zero(self):
        self.assertEqual(subtract(6, 0), 6)
"""
print(CALCULATOR)

## 4. Run the task with each harness

Only `profile.harness` changes. Each gets a separate directory and the same read, edit, and test tools; those tools can run commands in this Colab runtime.

In [ ]:
import asyncio
import time

from liteagents import ProfileOptions, run

profile = ProfileOptions(
    harness="pydantic-ai",
    model="openai/gpt-5.4-mini",
)
profile.tools = ["read_file", "edit_file", "run_tests"]
harnesses = ["pydantic-ai", "claude-sdk"]
prompt = "Read calculator.py and test_calculator.py. Fix the bug without changing the tests, then run them."
results = []
for harness in harnesses:
    work = Path(tempfile.mkdtemp(prefix=f"{harness}-", dir=workspace))
    (work / "calculator.py").write_text(CALCULATOR)
    (work / "test_calculator.py").write_text(TESTS)
    profile.harness = harness
    started = time.monotonic()
    report = {"harness": harness, "status": "failed", "workspace": str(work)}
    try:
        async with asyncio.timeout(180):
            result = await run(prompt, profile=profile, cwd=work)
        report.update(status="completed", answer=result.text, usage=result.usage)
    except Exception as error:  # noqa: BLE001 - report each harness independently.
        report["error"] = type(error).__name__
    report["seconds"] = round(time.monotonic() - started, 2)
    results.append(report)
    print(harness, report["status"], report["seconds"], "seconds")

## 5. Verify the fixes ourselves

Run the original tests independently of the agent’s answer. A row only passes if the tests pass and the agent did not change them.

In [ ]:
import json
import subprocess
import sys

from IPython.display import Markdown, display

rows = ["| Harness | Agent | Tests | Seconds |", "| --- | --- | --- | ---: |"]
for report in results:
    work = Path(report["workspace"])
    test_file = work / "test_calculator.py"
    report["tests_modified"] = not test_file.exists() or test_file.read_text() != TESTS
    test_file.write_text(TESTS)
    try:
        checked = subprocess.run(
            [sys.executable, "-m", "unittest", "-v"], cwd=work,
            capture_output=True, text=True, timeout=30, check=False,
        )
        report["tests"] = {"exit_code": checked.returncode, "output": checked.stdout + checked.stderr}
    except subprocess.TimeoutExpired:
        report["tests"] = {"exit_code": None, "output": "Tests timed out."}
    passed = report["tests"]["exit_code"] == 0 and not report["tests_modified"]
    rows.append(
        f"| {report['harness']} | {report['status']} | {'Passed' if passed else 'Failed'} | {report['seconds']} |"
    )
display(Markdown("\n".join(rows)))
(workspace / "results.json").write_text(json.dumps(results, indent=2))

## 6. Look at what changed

Choose a result below to see its answer, code diff, and test output.

In [ ]:
import difflib

selected = results[0]  # Change to results[1] for the other harness.
fixed = Path(selected["workspace"]) / "calculator.py"
print(selected.get("answer", selected.get("error", "")))
print("".join(difflib.unified_diff(
    CALCULATOR.splitlines(True), (fixed.read_text() if fixed.exists() else "").splitlines(True),
    fromfile="before", tofile="after",
)))
print(selected["tests"]["output"])

Try another prompt or add an installed harness to `harnesses`. Rerun step 4 to create fresh workspaces.

The times include startup; native usage reports are not a standardized cost comparison. Download `results.json` from the workspace if you want to keep the results.

[Other model providers](https://github.com/BerriAI/liteagents/blob/main/docs/models.md) · [All cookbooks](https://github.com/BerriAI/liteagents/blob/main/cookbook/README.md)